# Enhanced Numerai Model - Practical Implementation
This notebook improves upon the basic tutorial with ensemble methods, feature engineering, and better evaluation.

In [ ]:
# Install required packages
!pip install -q --upgrade numerapi pandas pyarrow matplotlib lightgbm xgboost scikit-learn scipy cloudpickle optuna
!pip install -q --no-deps numerai-tools

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# ML Libraries
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr
import cloudpickle

# Numerai
from numerapi import NumerAPI
napi = NumerAPI()

print("Setup complete!")

## 1. Data Loading with Memory Optimization

In [ ]:
DATA_VERSION = "v5.0"

# Download only if not exists
def download_if_needed(filename):
    filepath = Path(filename)
    if not filepath.exists():
        print(f"Downloading {filename}...")
        napi.download_dataset(filename)
    else:
        print(f"{filename} already exists, skipping download.")

# Download necessary files
download_if_needed(f"{DATA_VERSION}/features.json")
download_if_needed(f"{DATA_VERSION}/train.parquet")
download_if_needed(f"{DATA_VERSION}/validation.parquet")

In [ ]:
# Load feature metadata
with open(f"{DATA_VERSION}/features.json", 'r') as f:
    feature_metadata = json.load(f)

# Display available feature sets and targets
print(f"Available feature sets: {list(feature_metadata['feature_sets'].keys())}")
print(f"Number of targets: {len(feature_metadata['targets'])}")
print(f"Sample targets: {list(feature_metadata['targets'])[:5]}")

## 2. Feature Engineering Functions

In [ ]:
def engineer_features(df, feature_cols, create_interactions=True):
    """
    Engineer new features from raw features
    """
    features = df[feature_cols].copy()
    
    # 1. Statistical aggregations
    features['feature_mean'] = df[feature_cols].mean(axis=1)
    features['feature_std'] = df[feature_cols].std(axis=1)
    features['feature_skew'] = df[feature_cols].skew(axis=1)
    features['feature_kurt'] = df[feature_cols].kurtosis(axis=1)
    
    # 2. Percentile features
    features['feature_25p'] = df[feature_cols].quantile(0.25, axis=1)
    features['feature_75p'] = df[feature_cols].quantile(0.75, axis=1)
    features['feature_iqr'] = features['feature_75p'] - features['feature_25p']
    
    # 3. Feature interactions (limited to avoid memory issues)
    if create_interactions and len(feature_cols) > 10:
        # Select top features based on variance
        variances = df[feature_cols].var()
        top_features = variances.nlargest(5).index.tolist()
        
        for i, feat1 in enumerate(top_features[:-1]):
            for feat2 in top_features[i+1:i+2]:  # Limited interactions
                features[f'{feat1}_X_{feat2}'] = df[feat1] * df[feat2]
                features[f'{feat1}_DIV_{feat2}'] = df[feat1] / (df[feat2] + 1e-6)
    
    # 4. Rank features (within era if available)
    if 'era' in df.columns:
        for feat in feature_cols[:3]:  # Top 3 features only
            features[f'{feat}_rank'] = df.groupby('era')[feat].rank(pct=True)
    
    return features

print("Feature engineering functions defined!")

## 3. Model Training with Ensemble

In [ ]:
class NumeraiEnsemble:
    def __init__(self, feature_set='medium'):
        self.feature_set = feature_set
        self.models = {}
        self.weights = {}
        self.feature_cols = None
        
    def prepare_data(self, df, target_col=None, engineer=True):
        """Prepare features and target"""
        if engineer:
            X = engineer_features(df, self.feature_cols, create_interactions=True)
        else:
            X = df[self.feature_cols]
        
        y = df[target_col] if target_col else None
        return X, y
    
    def train(self, train_data, val_data, target_col='target_cyrus_v4_20'):
        """Train ensemble models"""
        # Set feature columns
        self.feature_cols = feature_metadata['feature_sets'][self.feature_set]
        
        # Prepare data
        print("Preparing training data...")
        X_train, y_train = self.prepare_data(train_data, target_col)
        X_val, y_val = self.prepare_data(val_data, target_col)
        
        print(f"Training shape: {X_train.shape}")
        print(f"Validation shape: {X_val.shape}")
        
        # 1. LightGBM
        print("\nTraining LightGBM...")
        self.models['lgb'] = lgb.LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.01,
            max_depth=6,
            num_leaves=31,
            colsample_bytree=0.7,
            subsample=0.7,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            verbose=-1
        )
        self.models['lgb'].fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        
        # 2. XGBoost
        print("Training XGBoost...")
        self.models['xgb'] = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.01,
            max_depth=6,
            colsample_bytree=0.7,
            subsample=0.7,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            verbosity=0
        )
        self.models['xgb'].fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=100,
            verbose=False
        )
        
        # 3. Random Forest
        print("Training Random Forest...")
        self.models['rf'] = RandomForestRegressor(
            n_estimators=100,
            max_depth=8,
            min_samples_split=100,
            min_samples_leaf=50,
            random_state=42,
            n_jobs=-1
        )
        self.models['rf'].fit(X_train, y_train)
        
        # 4. Ridge Regression
        print("Training Ridge Regression...")
        self.models['ridge'] = Ridge(alpha=10.0, random_state=42)
        self.models['ridge'].fit(X_train, y_train)
        
        # Calculate weights based on validation performance
        self.calculate_weights(X_val, y_val)
        
    def calculate_weights(self, X_val, y_val):
        """Calculate ensemble weights based on validation correlation"""
        correlations = {}
        
        for name, model in self.models.items():
            pred = model.predict(X_val)
            corr = spearmanr(pred, y_val)[0]
            correlations[name] = max(0, corr)  # Only positive correlations
            print(f"{name} validation correlation: {corr:.4f}")
        
        # Normalize weights
        total = sum(correlations.values())
        self.weights = {k: v/total for k, v in correlations.items()}
        print(f"\nEnsemble weights: {self.weights}")
        
    def predict(self, df):
        """Make ensemble predictions"""
        X, _ = self.prepare_data(df)
        predictions = np.zeros(len(X))
        
        for name, model in self.models.items():
            weight = self.weights.get(name, 0)
            if weight > 0:
                pred = model.predict(X)
                predictions += weight * pred
                
        return predictions

## 4. Training the Ensemble Model

In [ ]:
# Load data with selected columns only (memory efficient)
feature_set = 'medium'
feature_cols = feature_metadata['feature_sets'][feature_set]
target = 'target_cyrus_v4_20'

print(f"Loading training data with {len(feature_cols)} features...")
train_data = pd.read_parquet(
    f"{DATA_VERSION}/train.parquet",
    columns=['era'] + feature_cols + [target]
)

print(f"Loading validation data...")
val_data = pd.read_parquet(
    f"{DATA_VERSION}/validation.parquet",
    columns=['era'] + feature_cols + [target]
)

print(f"Train shape: {train_data.shape}")
print(f"Validation shape: {val_data.shape}")

In [ ]:
# Initialize and train ensemble
ensemble = NumeraiEnsemble(feature_set=feature_set)
ensemble.train(train_data, val_data, target_col=target)

## 5. Advanced Evaluation

In [ ]:
def evaluate_model(predictions, targets, eras):
    """Comprehensive model evaluation"""
    results = {}
    
    # Overall correlation
    results['overall_corr'] = spearmanr(predictions, targets)[0]
    
    # Era-wise correlations
    era_corrs = []
    for era in eras.unique():
        era_mask = eras == era
        if era_mask.sum() > 20:  # Minimum samples
            corr = spearmanr(predictions[era_mask], targets[era_mask])[0]
            era_corrs.append(corr)
    
    results['mean_corr'] = np.mean(era_corrs)
    results['std_corr'] = np.std(era_corrs)
    results['sharpe'] = results['mean_corr'] / (results['std_corr'] + 1e-6)
    
    # Max drawdown
    cumsum = np.cumsum(era_corrs)
    running_max = np.maximum.accumulate(cumsum)
    drawdown = cumsum - running_max
    results['max_drawdown'] = np.min(drawdown)
    
    # Consistency
    results['positive_eras'] = np.mean([c > 0 for c in era_corrs])
    
    return results, era_corrs

# Evaluate ensemble
val_predictions = ensemble.predict(val_data)
results, era_corrs = evaluate_model(val_predictions, val_data[target], val_data['era'])

print("\n" + "="*50)
print("VALIDATION PERFORMANCE METRICS")
print("="*50)
for metric, value in results.items():
    print(f"{metric:15s}: {value:.4f}")

In [ ]:
# Visualize performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Era correlations over time
axes[0, 0].plot(era_corrs)
axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Era Correlations Over Time')
axes[0, 0].set_xlabel('Era')
axes[0, 0].set_ylabel('Correlation')

# Cumulative returns
axes[0, 1].plot(np.cumsum(era_corrs))
axes[0, 1].set_title('Cumulative Performance')
axes[0, 1].set_xlabel('Era')
axes[0, 1].set_ylabel('Cumulative Correlation')

# Distribution of era correlations
axes[1, 0].hist(era_corrs, bins=50, edgecolor='black')
axes[1, 0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Distribution of Era Correlations')
axes[1, 0].set_xlabel('Correlation')
axes[1, 0].set_ylabel('Frequency')

# Rolling Sharpe
window = 20
rolling_mean = pd.Series(era_corrs).rolling(window).mean()
rolling_std = pd.Series(era_corrs).rolling(window).std()
rolling_sharpe = rolling_mean / (rolling_std + 1e-6)
axes[1, 1].plot(rolling_sharpe)
axes[1, 1].set_title(f'Rolling Sharpe Ratio (window={window})')
axes[1, 1].set_xlabel('Era')
axes[1, 1].set_ylabel('Sharpe Ratio')

plt.tight_layout()
plt.show()

## 6. Feature Neutralization (Risk Reduction)

In [ ]:
def neutralize_predictions(predictions, features, proportion=0.5):
    """
    Neutralize predictions to reduce feature exposure
    """
    from sklearn.preprocessing import StandardScaler
    from scipy.stats import rankdata
    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Calculate exposures
    exposures = np.linalg.lstsq(features_scaled, predictions, rcond=None)[0]
    
    # Remove proportion of exposure
    neutralized = predictions - proportion * features_scaled.dot(exposures)
    
    # Rank transform to maintain distribution
    neutralized = rankdata(neutralized, method='ordinal') / len(neutralized)
    neutralized = neutralized * predictions.std() + predictions.mean()
    
    return neutralized

# Apply neutralization
val_predictions_neutral = neutralize_predictions(
    val_predictions, 
    val_data[feature_cols],
    proportion=0.3
)

# Evaluate neutralized predictions
results_neutral, _ = evaluate_model(
    val_predictions_neutral, 
    val_data[target], 
    val_data['era']
)

print("\n" + "="*50)
print("NEUTRALIZED PREDICTIONS PERFORMANCE")
print("="*50)
for metric, value in results_neutral.items():
    print(f"{metric:15s}: {value:.4f}")

print("\n" + "="*50)
print("COMPARISON (Neutralized vs Raw)")
print("="*50)
for metric in results.keys():
    diff = results_neutral[metric] - results[metric]
    sign = '+' if diff > 0 else ''
    print(f"{metric:15s}: {sign}{diff:.4f}")

## 7. Create Submission Pipeline

In [ ]:
def create_prediction_pipeline(ensemble_model, feature_set, neutralize=True):
    """
    Create a complete prediction pipeline for deployment
    """
    def predict(live_features: pd.DataFrame, _live_benchmark_models: pd.DataFrame = None) -> pd.DataFrame:
        # Make predictions
        predictions = ensemble_model.predict(live_features)
        
        # Apply neutralization if requested
        if neutralize:
            feature_cols = ensemble_model.feature_cols
            predictions = neutralize_predictions(
                predictions,
                live_features[feature_cols],
                proportion=0.3
            )
        
        # Format submission
        submission = pd.Series(predictions, index=live_features.index)
        return submission.to_frame("prediction")
    
    return predict

# Create the pipeline
prediction_pipeline = create_prediction_pipeline(ensemble, feature_set, neutralize=True)

# Save the model
with open("enhanced_ensemble_model.pkl", "wb") as f:
    cloudpickle.dump(prediction_pipeline, f)

print("Model saved as enhanced_ensemble_model.pkl")
print("Ready for upload to Numerai!")

## 8. Test on Live Data

In [ ]:
# Download and test on live data
download_if_needed(f"{DATA_VERSION}/live.parquet")

live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_cols)
live_predictions = prediction_pipeline(live_features)

print(f"Live predictions shape: {live_predictions.shape}")
print(f"\nSample predictions:")
print(live_predictions.head())
print(f"\nPrediction statistics:")
print(live_predictions.describe())